# Magnetic Dipole Field Simulator
Interactive demo of the C++ dipole localisation pipeline via pybind11.

**Pipeline:** define dipole → place sensors → add noise → run LM solver → visualise

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'build'))
import dipole_py
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
print('dipole_py loaded ✓')

## 1. Define the true dipole

In [ ]:
truth = dipole_py.DipoleSource(
    dipole_py.Vec3(0.05, -0.03, 0.02),   # position [m]
    dipole_py.Vec3(0.0,   0.0,  1.5)     # moment   [A·m²]
)
print(f'Position : ({truth.position.x:.3f}, {truth.position.y:.3f}, {truth.position.z:.3f}) m')
print(f'Moment   : ({truth.moment.x:.3f}, {truth.moment.y:.3f}, {truth.moment.z:.3f}) A·m²')

## 2. Build sensor array and simulate noisy measurements

In [ ]:
array = dipole_py.SensorArray.ring(16, 0.08, 0.10)
noise = dipole_py.GaussianNoise(sigma=1e-9, seed=42)
meas  = array.measureNoisy(truth, noise)
print(f'{array.size()} sensors, noise σ = 1 nT')

# Field magnitude at first sensor
B0 = truth.fieldAt(array.sensors[0].position)
print(f'|B| at sensor 0 : {B0.norm()*1e6:.2f} μT')

## 3. Visualise sensor array and field vectors

In [ ]:
fig = plt.figure(figsize=(8, 6))
ax  = fig.add_subplot(111, projection='3d')

sx = [s.position.x for s in array.sensors]
sy = [s.position.y for s in array.sensors]
sz = [s.position.z for s in array.sensors]
ax.scatter(sx, sy, sz, c='steelblue', s=60, zorder=3, label='Sensors')

scale = 3e4
for s, b in zip(array.sensors, meas):
    ax.quiver(s.position.x, s.position.y, s.position.z,
              b.x*scale, b.y*scale, b.z*scale,
              color='coral', alpha=0.7, linewidth=0.8)

ax.scatter([truth.position.x],[truth.position.y],[truth.position.z],
           c='red', s=200, marker='*', zorder=5, label='Dipole')
ax.set_xlabel('x (m)'); ax.set_ylabel('y (m)'); ax.set_zlabel('z (m)')
ax.set_title('Sensor Array & B-field Vectors')
ax.legend(); plt.tight_layout(); plt.show()

## 4. Run Levenberg-Marquardt solver

In [ ]:
init   = dipole_py.DipoleSource(dipole_py.Vec3(0,0,0), dipole_py.Vec3(0,0,1))
result = dipole_py.LMSolver().solve(array, meas, init)

ep = result.estimated.position
pos_err = ((truth.position.x-ep.x)**2 +
           (truth.position.y-ep.y)**2 +
           (truth.position.z-ep.z)**2)**0.5 * 1e3

print(f'Converged  : {result.converged}')
print(f'Iterations : {result.iterations}')
print(f'Final cost : {result.finalCost:.2e}')
print(f'Pos error  : {pos_err:.4f} mm')

## 5. Monte Carlo: position error vs noise level

In [ ]:
sigmas  = np.logspace(-11, -6, 6)
n_trials = 200
solver   = dipole_py.LMSolver()
means, stds = [], []

for sigma in sigmas:
    errs = []
    for seed in range(n_trials):
        n  = dipole_py.GaussianNoise(sigma, seed)
        m  = array.measureNoisy(truth, n)
        r  = solver.solve(array, m, init)
        p  = r.estimated.position
        errs.append(((truth.position.x-p.x)**2 +
                     (truth.position.y-p.y)**2 +
                     (truth.position.z-p.z)**2)**0.5 * 1e3)
    means.append(np.mean(errs))
    stds.append(np.std(errs))
    print(f'σ={sigma:.0e} T → mean error {means[-1]:.4f} mm')

plt.figure(figsize=(7,4))
plt.errorbar(sigmas, means, yerr=stds, fmt='o-', capsize=5, color='steelblue')
plt.xscale('log'); plt.yscale('log')
plt.xlabel('Noise σ (T)'); plt.ylabel('Mean position error (mm)')
plt.title('Position Error vs Noise Level (200 trials each)')
plt.grid(True, which='both', ls='--', alpha=0.4)
plt.tight_layout(); plt.show()